# 02 - Feature engineering and EDA
Uses the same `engineer_features` as the training script (UTC index, local-time calendar features, no `pct_change`),
then plots basic behaviour for every building in `BUILDINGS`.

Reads `data/interim/<building>_interim.csv` (run notebook 01 first). Writes `data/interim/<building>_features.csv`
(kept out of `data/processed/` so it cannot be confused with the training outputs).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
for p in (PROJECT_ROOT, PROJECT_ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.config import BUILDINGS, CONFIG, FEATURE_COLS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.feature_engineering import engineer_features

META_PATH = PROJECT_ROOT / CONFIG["meta_url"]
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"

meta = pd.read_csv(META_PATH)
meta[meta["uid"].isin(BUILDINGS)][["uid", "timezone", "primaryspaceuse_abbrev", "sqm"]]

### Build features for every configured building

In [ ]:
features = {}

for building in BUILDINGS:
    df = pd.read_csv(INTERIM_PATH / f"{building}_interim.csv")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.set_index("timestamp").sort_index()

    feat = engineer_features(df, building, meta)
    feat.to_csv(INTERIM_PATH / f"{building}_features.csv")
    features[building] = feat

    idx = feat.index.to_series().diff().dropna()
    print(f"{building}: shape={feat.shape} dup_ts={feat.index.duplicated().sum()} "
          f"non-hourly gaps={(idx != pd.Timedelta('1h')).sum()}")

### Sanity checks that the engineered features behave

In [ ]:
# No NaN / inf, strictly hourly, and the feature list used by the model is present
for building, feat in features.items():
    assert set(FEATURE_COLS) <= set(feat.columns), building
    assert not feat[FEATURE_COLS].isna().any().any(), building
    assert np.isfinite(feat[FEATURE_COLS].to_numpy()).all(), building
print("Feature checks passed for", len(features), "buildings; n_features =", len(FEATURE_COLS))

### Energy over time (+ 24h rolling mean)

In [ ]:
fig, axes = plt.subplots(len(BUILDINGS), 1, figsize=(12, 3 * len(BUILDINGS)), sharex=False)
for ax, (building, feat) in zip(axes, features.items()):
    feat[["energy", "rolling_mean_24"]].plot(ax=ax, lw=0.8, title=building)
    ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Energy distribution

In [ ]:
fig, axes = plt.subplots(1, len(BUILDINGS), figsize=(4 * len(BUILDINGS), 3))
for ax, (building, feat) in zip(axes, features.items()):
    sns.histplot(feat["energy"], bins=50, kde=True, ax=ax)
    ax.set_title(building)
plt.tight_layout(); plt.show()

### Average hourly profile (building-local hour) and day-of-week profile

In [ ]:
def local_calendar(feat, building):
    tz = meta.loc[meta["uid"] == building, "timezone"].values[0]
    local = feat.index.tz_convert(tz)
    return pd.DataFrame({"hour": local.hour, "dow": local.dayofweek, "energy": feat["energy"].to_numpy()})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for building, feat in features.items():
    cal = local_calendar(feat, building)
    # normalise by each building's mean so the shapes are comparable
    norm = cal["energy"] / cal["energy"].mean()
    norm.groupby(cal["hour"]).mean().plot(ax=axes[0], marker="o", label=building)
    norm.groupby(cal["dow"]).mean().plot(ax=axes[1], marker="o", label=building)
axes[0].set_title("Average hourly profile (relative to building mean)"); axes[0].set_xlabel("local hour")
axes[1].set_title("Average day-of-week profile (0 = Monday)"); axes[1].set_xlabel("day of week")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

### Train/test coverage check
With about one year of data per building, the chronological 80/20 split puts the last ~2-3 months in the test set.
This shows which months each side sees (the first version used a raw `month` feature, which was out of range in the test period).

In [ ]:
rows = []
for building, feat in features.items():
    si = int(len(feat) * (1 - CONFIG["test_split"]))
    tr, te = feat.iloc[:si], feat.iloc[si:]
    rows.append({"building": building, "split_at_utc": feat.index[si],
                 "train_months": sorted(set(tr.index.month)), "test_months": sorted(set(te.index.month)),
                 "test_energy_above_train_max": round(float((te["energy"] > tr["energy"].max()).mean()), 4)})
pd.DataFrame(rows)